# Fetaure Engineering

## 1. Dimensionality & Data Structure

- **LabelEncoder** - For 1D arrays (single column/labels)
- **OrdinalEncoder** - For 2D arrays (multiple columns/features)

In [8]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import numpy as np

# Sample data
y = np.array(['cat', 'dog', 'bird', 'dog', 'cat'])  # 1D - target/labels
X = np.array([['cat', 'high'], 
              ['dog', 'medium'], 
              ['bird', 'low']])  # 2D - features

# LabelEncoder - works with 1D
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"LabelEncoder output (1D): {y_encoded}")
print(f"Shape: {y_encoded.shape}")

# OrdinalEncoder - works with 2D
oe = OrdinalEncoder()
X_encoded = oe.fit_transform(X)
print(f"\nOrdinalEncoder output (2D): {X_encoded}")
print(f"Shape: {X_encoded.shape}")

LabelEncoder output (1D): [1 2 0 2 1]
Shape: (5,)

OrdinalEncoder output (2D): [[1. 0.]
 [2. 2.]
 [0. 1.]]
Shape: (3, 2)


## 2. Primary Use Case
- **LabelEncoder** - Primarily for encoding target variables (y) in classification
- **OrdinalEncoder** - For encoding feature variables (X) with ordinal categorical data

In [9]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import pandas as pd

# Typical usage scenario
df = pd.DataFrame({
    'target': ['spam', 'ham', 'spam', 'ham'],  # Target variable
    'size': ['small', 'medium', 'large', 'medium'],  # Feature 1
    'priority': ['low', 'high', 'medium', 'high']    # Feature 2
})

# LabelEncoder for target
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df['target'])

# OrdinalEncoder for features (with specified order!)
oe = OrdinalEncoder(categories=[['small', 'medium', 'large'],  # size has order
                                 ['low', 'medium', 'high']])    # priority has order
features_encoded = oe.fit_transform(df[['size', 'priority']])

print("LabelEncoder (target):")
print(df[['target', 'target_encoded']].drop_duplicates())

print("\nOrdinalEncoder (features with order preserved):")
print(features_encoded)
print("Categories order:", oe.categories_)

LabelEncoder (target):
  target  target_encoded
0   spam               1
1    ham               0

OrdinalEncoder (features with order preserved):
[[0. 0.]
 [1. 2.]
 [2. 1.]
 [1. 2.]]
Categories order: [array(['small', 'medium', 'large'], dtype=object), array(['low', 'medium', 'high'], dtype=object)]


## 3. Order Specification Capability

- **LabelEncoder** - No built-in way to specify order
- **OrdinalEncoder** - Can explicitly specify order with `categories` parameter

In [10]:
from sklearn.preprocessing import OrdinalEncoder

# OrdinalEncoder can preserve meaningful order
education_levels = [['PhD', 'Masters', 'Bachelor', 'High School']]
income_levels = [['high', 'medium', 'low']]

oe = OrdinalEncoder(categories=education_levels + income_levels)

data = [['Masters', 'medium'],
        ['Bachelor', 'low'],
        ['PhD', 'high']]

encoded = oe.fit_transform(data)
print("OrdinalEncoder with meaningful order:")
print(encoded)
print("0=PhD, 1=Masters, 2=Bachelor, 3=High School")

OrdinalEncoder with meaningful order:
[[1. 1.]
 [2. 2.]
 [0. 0.]]
0=PhD, 1=Masters, 2=Bachelor, 3=High School


# 4. Unknown Categories Handling

- **OrdinalEncoder** - Has handle_unknown parameter
- **LabelEncoder** - No built-in unknown category handling

In [11]:
# OrdinalEncoder can handle unknown categories during transform
oe = OrdinalEncoder(handle_unknown='use_encoded_value', 
                    unknown_value=-1)

train_data = [['cat'], ['dog'], ['bird']]
test_data = [['cat'], ['dog'], ['unknown_animal']]  # Has unseen category

oe.fit(train_data)
print("Train transform:", oe.transform(train_data))
print("Test transform (unknown becomes -1):", oe.transform(test_data))

# LabelEncoder would raise an error with unseen categories
le = LabelEncoder()
le.fit(['cat', 'dog', 'bird'])
try:
    le.transform(['cat', 'unknown'])  # This will raise ValueError
except ValueError as e:
    print(f"\nLabelEncoder error: {e}")

Train transform: [[1.]
 [2.]
 [0.]]
Test transform (unknown becomes -1): [[ 1.]
 [ 2.]
 [-1.]]

LabelEncoder error: y contains previously unseen labels: np.str_('unkn')


# 5. Output Type

- **LabelEncoder** - Returns numpy array
- **OrdinalEncoder** - Returns numpy array (can be sparse)

In [12]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# Create sample dataset
data = {
    'target': ['yes', 'no', 'yes', 'maybe'],
    'feature1': ['small', 'medium', 'large', 'medium'],
    'feature2': ['low', 'high', 'medium', 'high']
}
df = pd.DataFrame(data)

print("Original Data:")
print(df)
print("\n" + "="*50 + "\n")

# When to use LabelEncoder
print("1. USE LabelEncoder FOR TARGET VARIABLES:")
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df['target'])
print(df[['target', 'target_encoded']].drop_duplicates().sort_values('target_encoded'))

print("\n" + "="*50 + "\n")

# When to use OrdinalEncoder
print("2. USE OrdinalEncoder FOR FEATURE VARIABLES:")
# Specify meaningful order for ordinal features
oe = OrdinalEncoder(categories=[
    ['small', 'medium', 'large'],  # feature1 has natural order
    ['low', 'medium', 'high']      # feature2 has natural order
])

features_encoded = oe.fit_transform(df[['feature1', 'feature2']])
df_encoded = pd.DataFrame(features_encoded, columns=['feature1_encoded', 'feature2_encoded'])
print(df_encoded)
print("\nEncoded values maintain ordinal relationship:")
print("small(0) < medium(1) < large(2)")
print("low(0) < medium(1) < high(2)")

Original Data:
  target feature1 feature2
0    yes    small      low
1     no   medium     high
2    yes    large   medium
3  maybe   medium     high


1. USE LabelEncoder FOR TARGET VARIABLES:
  target  target_encoded
3  maybe               0
1     no               1
0    yes               2


2. USE OrdinalEncoder FOR FEATURE VARIABLES:
   feature1_encoded  feature2_encoded
0               0.0               0.0
1               1.0               2.0
2               2.0               1.0
3               1.0               2.0

Encoded values maintain ordinal relationship:
small(0) < medium(1) < large(2)
low(0) < medium(1) < high(2)


# Complete Side-by-Side Example:

In [13]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import numpy as np
import pandas as pd

# Create comprehensive example
np.random.seed(42)
n_samples = 10

data = pd.DataFrame({
    'target': np.random.choice(['spam', 'ham', 'unknown'], n_samples),
    'size': np.random.choice(['XS', 'S', 'M', 'L', 'XL'], n_samples),
    'priority': np.random.choice(['low', 'medium', 'high'], n_samples),
    'category': np.random.choice(['A', 'B', 'C', 'D'], n_samples)
})

print("ORIGINAL DATA:")
print(data)
print("\n" + "="*60 + "\n")

# Process with LabelEncoder (target only)
le = LabelEncoder()
data['target_encoded'] = le.fit_transform(data['target'])

# Process with OrdinalEncoder (features)
# Define order where it matters
oe = OrdinalEncoder(categories=[
    ['XS', 'S', 'M', 'L', 'XL'],    # size has ordinal relationship
    ['low', 'medium', 'high'],      # priority has ordinal relationship
    ['A', 'B', 'C', 'D']            # category is nominal but we want consistent order
])

feature_cols = ['size', 'priority', 'category']
features_encoded = oe.fit_transform(data[feature_cols])
encoded_df = pd.DataFrame(features_encoded, 
                          columns=[f'{col}_encoded' for col in feature_cols])

result = pd.concat([data, encoded_df], axis=1)
print("PROCESSED DATA:")
print(result[['target', 'target_encoded', 
              'size', 'size_encoded',
              'priority', 'priority_encoded',
              'category', 'category_encoded']])

print("\n" + "="*60 + "\n")
print("ENCODING MAPPINGS:")
print(f"Target mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")
for i, col in enumerate(feature_cols):
    print(f"{col} mapping: {dict(zip(oe.categories_[i], range(len(oe.categories_[i]))))}")

ORIGINAL DATA:
    target size priority category
0  unknown    M      low        B
1     spam    M      low        D
2  unknown   XL   medium        D
3  unknown    L   medium        D
4     spam    M      low        D
5     spam   XL      low        C
6  unknown    S      low        B
7      ham    L     high        B
8  unknown    S     high        C
9  unknown    L     high        B


PROCESSED DATA:
    target  target_encoded size  size_encoded priority  priority_encoded  \
0  unknown               2    M           2.0      low               0.0   
1     spam               1    M           2.0      low               0.0   
2  unknown               2   XL           4.0   medium               1.0   
3  unknown               2    L           3.0   medium               1.0   
4     spam               1    M           2.0      low               0.0   
5     spam               1   XL           4.0      low               0.0   
6  unknown               2    S           1.0      low       

# When to Use Which:

## Use LabelEncoder when:

- Encoding target/label variables for classification

- Working with 1D arrays (single column)

- You don't need to specify custom order

- Simple label encoding without unknown category handling

## Use OrdinalEncoder when:

- Encoding feature variables (multiple columns)

- Features have ordinal relationships (small < medium < large)

- You need to specify custom order of categories

- You need to handle unknown categories during inference

- Working with 2D data (DataFrames, feature matrices)

## Modern Best Practice:

In [14]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer

# For modern sklearn pipelines, use OrdinalEncoder even for single features
# This is more consistent and handles edge cases better

# Instead of mixing LabelEncoder and OrdinalEncoder:
ct = ColumnTransformer([
    ('ordinal', OrdinalEncoder(categories=[['small', 'medium', 'large']]), ['size']),
    ('ordinal2', OrdinalEncoder(categories=[['low', 'medium', 'high']]), ['priority'])
], remainder='passthrough')

# This approach is more robust for production pipelines

## Key Takeaway:

- **LabelEncoder** = "I'm encoding target labels (y)"
- **OrdinalEncoder** = "I'm encoding feature columns (X) with ordinal categories"

# CustomOrdinalEncoder

In [15]:
import pandas as pd
import numpy as np
from typing import List, Optional, Union, Dict
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder
import warnings

class CustomOrdinalEncoder(BaseEstimator, TransformerMixin):
    """
    Custom ordinal encoder that supports both automatic and manual encoding.
    
    This transformer applies ordinal encoding to specified columns with options for:
    1. Automatic encoding (based on data order)
    2. Manual encoding with custom order (using dictionary mapping)
    3. Return either pandas DataFrame or numpy array
    
    Parameters
    ----------
    columns : List[str]
        The list of column names to apply ordinal encoding to.
    
    custom_mapping : Optional[Union[Dict[str, List], Dict[str, Dict]]], default=None
        Custom mapping for ordinal encoding. Can be either:
        - Dictionary of lists: {'col': ['cat1', 'cat2', 'cat3']}
          (order determines encoding: 'cat1'=0, 'cat2'=1, etc.)
        - Dictionary of dictionaries: {'col': {'cat1': 0, 'cat2': 1, 'cat3': 2}}
          (explicit mapping)
    
    handle_unknown : str, default='error'
        How to handle unknown categories during transform:
        - 'error': raise an error
        - 'use_encoded_value': use the value specified in `unknown_value`
        - 'ignore': return NaN/None
    
    unknown_value : int or None, default=None
        Value to use for unknown categories when handle_unknown='use_encoded_value'.
        If None, will use n_categories (assigns new category at the end).
    
    return_array : bool, default=False
        If True, returns numpy array. If False, returns pandas DataFrame.
    
    categories : str, default='auto'
        Categories to use for encoding (only used when custom_mapping=None).
        - 'auto': Determine categories from the training data
        - List of arrays: Specify categories for each column
    
    Attributes
    ----------
    encoder_ : sklearn.preprocessing.OrdinalEncoder
        The underlying sklearn OrdinalEncoder instance.
    
    feature_names_in_ : np.ndarray
        Names of features seen during fit.
    
    categories_ : list
        Categories for each feature determined during fitting.
    
    n_features_in_ : int
        Number of features seen during fit.
    
    mapping_ : dict
        Final mapping used for encoding (for reference).
    
    Notes
    -----
    Author  : Rahul Shelke
    Created : 2025-07-23 
    Modified: Enhanced for production use
    
    Examples
    --------
    >>> encoder = CustomOrdinalEncoder(columns=['education', 'size'])
    >>> encoder.fit(X_train)
    >>> X_encoded = encoder.transform(X_test)
    
    >>> # With custom mapping (dictionary of lists)
    >>> custom_map = {'education': ['High School', 'Bachelor', 'Masters', 'PhD']}
    >>> encoder = CustomOrdinalEncoder(columns=['education'], custom_mapping=custom_map)
    
    >>> # With custom mapping (dictionary of dictionaries)
    >>> custom_map = {'education': {'PhD': 0, 'Masters': 1, 'Bachelor': 2, 'HS': 3}}
    >>> encoder = CustomOrdinalEncoder(columns=['education'], custom_mapping=custom_map)
    """
    
    def __init__(
        self,
        columns: List[str],
        custom_mapping: Optional[Union[Dict[str, List], Dict[str, Dict]]] = None,
        handle_unknown: str = 'error',
        unknown_value: Optional[int] = None,
        return_array: bool = False,
        categories: str = 'auto'
    ):
        self.columns = columns
        self.custom_mapping = custom_mapping
        self.handle_unknown = handle_unknown
        self.unknown_value = unknown_value
        self.return_array = return_array
        self.categories = categories
        
        self._validate_parameters()
    
    def _validate_parameters(self):
        """Validate initialization parameters."""
        if not isinstance(self.columns, list):
            raise TypeError(f"columns must be a list, got {type(self.columns)}")
        
        if self.custom_mapping is not None:
            if not isinstance(self.custom_mapping, dict):
                raise TypeError(f"custom_mapping must be a dict, got {type(self.custom_mapping)}")
            
            # Validate custom mapping structure
            for col, mapping in self.custom_mapping.items():
                if col not in self.columns:
                    warnings.warn(f"Column '{col}' in custom_mapping not in columns list")
                
                if isinstance(mapping, list):
                    # Validate list mapping
                    if len(mapping) != len(set(mapping)):
                        raise ValueError(f"Duplicate values found in mapping for column '{col}'")
                elif isinstance(mapping, dict):
                    # Validate dict mapping
                    values = list(mapping.values())
                    if len(values) != len(set(values)):
                        raise ValueError(f"Duplicate encoded values found in mapping for column '{col}'")
                    if not all(isinstance(v, (int, np.integer)) for v in values):
                        raise ValueError(f"All mapping values must be integers for column '{col}'")
                else:
                    raise TypeError(f"Mapping for column '{col}' must be list or dict, got {type(mapping)}")
        
        if self.handle_unknown not in ['error', 'use_encoded_value', 'ignore']:
            raise ValueError(f"handle_unknown must be 'error', 'use_encoded_value', or 'ignore', got {self.handle_unknown}")
    
    def _prepare_categories_from_mapping(self, X: pd.DataFrame):
        """Prepare categories parameter from custom mapping."""
        if self.custom_mapping is None:
            return self.categories
        
        categories = []
        for col in self.columns:
            if col in self.custom_mapping:
                mapping = self.custom_mapping[col]
                if isinstance(mapping, list):
                    # For list mapping, use the list order as categories
                    # Validate all categories exist in data (for fitting)
                    unique_vals = X[col].dropna().unique()
                    if not set(mapping).issuperset(set(unique_vals)):
                        missing = set(unique_vals) - set(mapping)
                        warnings.warn(f"Column '{col}' has values not in custom mapping: {missing}")
                    categories.append(mapping)
                else:  # dict mapping
                    # For dict mapping, sort by value to get category order
                    sorted_items = sorted(mapping.items(), key=lambda x: x[1])
                    cat_order = [item[0] for item in sorted_items]
                    
                    # Validate mapping covers all values in data
                    unique_vals = X[col].dropna().unique()
                    if not set(mapping.keys()).issuperset(set(unique_vals)):
                        missing = set(unique_vals) - set(mapping.keys())
                        warnings.warn(f"Column '{col}' has values not in custom mapping: {missing}")
                    
                    categories.append(cat_order)
            else:
                # No custom mapping for this column, use 'auto'
                categories.append('auto')
        
        return categories
    
    def _create_final_mapping(self):
        """Create final mapping dictionary for reference."""
        self.mapping_ = {}
        for i, col in enumerate(self.columns):
            if hasattr(self.encoder_, 'categories_'):
                categories = self.encoder_.categories_[i]
                self.mapping_[col] = {cat: idx for idx, cat in enumerate(categories)}
    
    def fit(self, X: pd.DataFrame, y=None):
        """
        Fit the encoder to the data.
        
        Parameters
        ----------
        X : pd.DataFrame
            Input data to fit.
        y : None
            Ignored. Exists for compatibility.
        
        Returns
        -------
        self : CustomOrdinalEncoder
            Returns the instance itself.
        """
        # Validate input
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")
        
        missing_cols = [col for col in self.columns if col not in X.columns]
        if missing_cols:
            raise ValueError(f"Columns not found in X: {missing_cols}")
        
        # Store feature names
        self.feature_names_in_ = np.array(X.columns)
        self.n_features_in_ = X.shape[1]
        
        # Prepare categories based on custom mapping
        categories = self._prepare_categories_from_mapping(X)
        
        # Initialize and fit the encoder
        self.encoder_ = OrdinalEncoder(
            categories=categories,
            handle_unknown=self.handle_unknown,
            unknown_value=self.unknown_value,
            dtype=np.float64  # Use float to handle NaN
        )
        
        # Fit encoder on selected columns
        self.encoder_.fit(X[self.columns])
        
        # Store categories
        self.categories_ = self.encoder_.categories_
        
        # Create final mapping for reference
        self._create_final_mapping()
        
        return self
    
    def transform(self, X: pd.DataFrame) -> Union[pd.DataFrame, np.ndarray]:
        """
        Transform the data using the fitted encoder.
        
        Parameters
        ----------
        X : pd.DataFrame
            Input data to transform.
        
        Returns
        -------
        X_transformed : Union[pd.DataFrame, np.ndarray]
            Transformed data. Returns array if return_array=True, else DataFrame.
        """
        # Check if fit has been called
        if not hasattr(self, 'encoder_'):
            raise RuntimeError("Must call fit before transform")
        
        # Validate input
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")
        
        missing_cols = [col for col in self.columns if col not in X.columns]
        if missing_cols:
            raise ValueError(f"Columns not found in X: {missing_cols}")
        
        # Transform the data
        X_transformed_array = self.encoder_.transform(X[self.columns])
        
        if self.return_array:
            return X_transformed_array
        else:
            # Get feature names for output
            if hasattr(self.encoder_, 'get_feature_names_out'):
                feature_names = self.encoder_.get_feature_names_out(self.columns)
            else:
                # Fallback for older sklearn versions
                feature_names = self.columns
            
            # Create DataFrame with original index
            X_transformed = pd.DataFrame(
                X_transformed_array,
                columns=feature_names,
                index=X.index
            )
            
            # Preserve other columns
            other_cols = [col for col in X.columns if col not in self.columns]
            if other_cols:
                X_transformed = pd.concat([X_transformed, X[other_cols]], axis=1)
            
            return X_transformed
    
    def fit_transform(self, X: pd.DataFrame, y=None) -> Union[pd.DataFrame, np.ndarray]:
        """
        Fit and transform the data in one step.
        
        Parameters
        ----------
        X : pd.DataFrame
            Input data to fit and transform.
        y : None
            Ignored. Exists for compatibility.
        
        Returns
        -------
        X_transformed : Union[pd.DataFrame, np.ndarray]
            Transformed data.
        """
        return self.fit(X, y).transform(X)
    
    def inverse_transform(self, X: Union[pd.DataFrame, np.ndarray]) -> pd.DataFrame:
        """
        Convert encoded data back to original categories.
        
        Parameters
        ----------
        X : Union[pd.DataFrame, np.ndarray]
            Encoded data to inverse transform.
        
        Returns
        -------
        X_original : pd.DataFrame
            Data with original categories.
        """
        if not hasattr(self, 'encoder_'):
            raise RuntimeError("Must call fit before inverse_transform")
        
        # Handle input type
        if isinstance(X, pd.DataFrame):
            # Extract only the encoded columns
            X_array = X[self.columns].values if self.return_array else X[self.columns].values
            result_array = self.encoder_.inverse_transform(X_array)
        else:
            result_array = self.encoder_.inverse_transform(X)
        
        # Create DataFrame with original column names
        result_df = pd.DataFrame(result_array, columns=self.columns)
        
        # If input was DataFrame with additional columns, preserve them
        if isinstance(X, pd.DataFrame) and not self.return_array:
            other_cols = [col for col in X.columns if col not in self.columns]
            if other_cols:
                result_df = pd.concat([result_df, X[other_cols]], axis=1)
        
        return result_df
    
    def get_feature_names_out(self, input_features=None):
        """
        Get output feature names for transformation.
        
        Parameters
        ----------
        input_features : array-like of str or None, default=None
            Input features.
        
        Returns
        -------
        feature_names_out : ndarray of str objects
            Transformed feature names.
        """
        if not hasattr(self, 'encoder_'):
            raise RuntimeError("Must call fit before get_feature_names_out")
        
        if hasattr(self.encoder_, 'get_feature_names_out'):
            return self.encoder_.get_feature_names_out(self.columns)
        else:
            return np.array(self.columns)
    
    def get_params(self, deep=True):
        """Get parameters for this estimator."""
        params = {
            'columns': self.columns,
            'custom_mapping': self.custom_mapping,
            'handle_unknown': self.handle_unknown,
            'unknown_value': self.unknown_value,
            'return_array': self.return_array,
            'categories': self.categories
        }
        return params
    
    def set_params(self, **params):
        """Set the parameters of this estimator."""
        for key, value in params.items():
            setattr(self, key, value)
        self._validate_parameters()
        return self

# Usage

In [16]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Create custom dataset with various categorical scenarios
data = {
    'education': ['PhD', 'Masters', 'Bachelor', 'High School', 'PhD', 'Masters', 
                  'Bachelor', 'High School', 'Masters', 'PhD'],
    'size': ['Small', 'Medium', 'Large', 'Small', 'Medium', 'Large', 
             'Small', 'Medium', 'Large', np.nan],
    'satisfaction': ['Low', 'Medium', 'High', 'Low', 'Medium', 'High', 
                     'Low', 'Medium', 'High', 'Medium'],
    'department': ['HR', 'IT', 'Finance', 'IT', 'HR', 'Finance', 
                   'IT', 'HR', 'Finance', 'IT'],
    'salary': [150000, 120000, 90000, 60000, 140000, 110000, 
               80000, 55000, 100000, 70000],
    'experience': [10, 7, 5, 2, 9, 6, 4, 1, 8, 3]
}

df = pd.DataFrame(data)
print("Original Data:")
print(df)
print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 1: Basic Usage (Auto encoding)
# ============================================================================
print("EXAMPLE 1: Basic Auto Encoding")
print("-" * 40)

encoder1 = CustomOrdinalEncoder(
    columns=['education', 'size', 'satisfaction'],
    return_array=False
)

encoder1.fit(df)
df_encoded1 = encoder1.transform(df)
print("Encoded Data (Auto encoding):")
print(df_encoded1[['education', 'size', 'satisfaction']])
print("\nMapping used:")
for col, mapping in encoder1.mapping_.items():
    print(f"{col}: {mapping}")
print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 2: Custom Mapping with List Order
# ============================================================================
print("EXAMPLE 2: Custom Mapping with List Order")
print("-" * 40)

custom_list_mapping = {
    'education': ['High School', 'Bachelor', 'Masters', 'PhD'],  # Order matters!
    'satisfaction': ['Low', 'Medium', 'High']  # Natural order
}

encoder2 = CustomOrdinalEncoder(
    columns=['education', 'satisfaction', 'department'],
    custom_mapping=custom_list_mapping,
    return_array=False
)

df_encoded2 = encoder2.fit_transform(df)
print("Encoded Data (List mapping):")
print(df_encoded2[['education', 'satisfaction', 'department']])
print("\nMapping used:")
for col, mapping in encoder2.mapping_.items():
    print(f"{col}: {mapping}")
print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 3: Custom Mapping with Dictionary (Explicit values)
# ============================================================================
print("EXAMPLE 3: Custom Mapping with Dictionary (Explicit values)")
print("-" * 40)

custom_dict_mapping = {
    'education': {
        'High School': 3,  # Can assign any integer
        'Bachelor': 2,
        'Masters': 1,
        'PhD': 0           # PhD gets highest importance (0)
    },
    'size': {
        'Small': 0,
        'Medium': 1,
        'Large': 2
    }
}

encoder3 = CustomOrdinalEncoder(
    columns=['education', 'size'],
    custom_mapping=custom_dict_mapping,
    return_array=False
)

df_encoded3 = encoder3.fit_transform(df)
print("Encoded Data (Dict mapping):")
print(df_encoded3[['education', 'size']])
print("\nMapping used:")
for col, mapping in encoder3.mapping_.items():
    print(f"{col}: {mapping}")
print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 4: Handling Unknown Categories
# ============================================================================
print("EXAMPLE 4: Handling Unknown Categories")
print("-" * 40)

# Create test data with unseen category
test_data = {
    'education': ['PhD', 'Bachelor', 'Unknown Degree'],  # Unknown Degree not seen during fit
    'size': ['Small', 'Medium', 'Large'],
    'satisfaction': ['High', 'Medium', 'Low']
}
df_test = pd.DataFrame(test_data)

# Strategy 1: Error (default)
encoder4a = CustomOrdinalEncoder(
    columns=['education'],
    custom_mapping={'education': ['High School', 'Bachelor', 'Masters', 'PhD']},
    handle_unknown='error'
)
encoder4a.fit(df[['education']])

print("Strategy 1: handle_unknown='error'")
try:
    result = encoder4a.transform(df_test[['education']])
    print("Successfully transformed")
except Exception as e:
    print(f"Error (expected): {type(e).__name__}: {str(e)[:50]}...")

# Strategy 2: Use encoded value
encoder4b = CustomOrdinalEncoder(
    columns=['education'],
    custom_mapping={'education': ['High School', 'Bachelor', 'Masters', 'PhD']},
    handle_unknown='use_encoded_value',
    unknown_value=-1  # Assign -1 to unknown categories
)
encoder4b.fit(df[['education']])
result4b = encoder4b.transform(df_test[['education']])
print("\nStrategy 2: handle_unknown='use_encoded_value', unknown_value=-1")
print("Encoded education (Unknown Degree → -1):")
print(result4b)

# Strategy 3: Ignore (returns NaN)
encoder4c = CustomOrdinalEncoder(
    columns=['education'],
    custom_mapping={'education': ['High School', 'Bachelor', 'Masters', 'PhD']},
    handle_unknown='ignore'
)
encoder4c.fit(df[['education']])
result4c = encoder4c.transform(df_test[['education']])
print("\nStrategy 3: handle_unknown='ignore'")
print("Encoded education (Unknown Degree → NaN):")
print(result4c)
print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 5: Returning Numpy Array vs DataFrame
# ============================================================================
print("EXAMPLE 5: Numpy Array vs DataFrame Output")
print("-" * 40)

# Return as numpy array
encoder5a = CustomOrdinalEncoder(
    columns=['education', 'size'],
    return_array=True
)
array_result = encoder5a.fit_transform(df[['education', 'size']])
print("Numpy Array output (return_array=True):")
print(f"Type: {type(array_result)}")
print(f"Shape: {array_result.shape}")
print(f"First 3 rows:\n{array_result[:3]}")
print()

# Return as DataFrame
encoder5b = CustomOrdinalEncoder(
    columns=['education', 'size'],
    return_array=False
)
df_result = encoder5b.fit_transform(df[['education', 'size']])
print("DataFrame output (return_array=False):")
print(f"Type: {type(df_result)}")
print(f"Columns: {df_result.columns.tolist()}")
print(f"Index preserved: {df_result.index.equals(df.index)}")
print(f"First 3 rows:\n{df_result.head(3)}")
print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 6: Integration with sklearn Pipeline
# ============================================================================
print("EXAMPLE 6: Integration with sklearn Pipeline")
print("-" * 40)

# Create a simple classification dataset
np.random.seed(42)
n_samples = 100
X = pd.DataFrame({
    'education': np.random.choice(['HS', 'Bachelor', 'Masters', 'PhD'], n_samples),
    'size': np.random.choice(['Small', 'Medium', 'Large'], n_samples),
    'department': np.random.choice(['HR', 'IT', 'Finance'], n_samples),
    'salary': np.random.normal(70000, 20000, n_samples),
    'experience': np.random.exponential(5, n_samples)
})
y = (X['salary'] > 70000).astype(int)  # Binary target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define custom mapping for the pipeline
pipeline_mapping = {
    'education': ['HS', 'Bachelor', 'Masters', 'PhD'],
    'size': {'Small': 0, 'Medium': 1, 'Large': 2}
}

# Create pipeline with CustomOrdinalEncoder
pipeline = Pipeline([
    ('encoder', CustomOrdinalEncoder(
        columns=['education', 'size', 'department'],
        custom_mapping=pipeline_mapping,
        handle_unknown='use_encoded_value',
        unknown_value=-1,
        return_array=True  # Most sklearn estimators expect arrays
    )),
    ('classifier', RandomForestClassifier(n_estimators=10, random_state=42))
])

# Train and evaluate
pipeline.fit(X_train, y_train)
score = pipeline.score(X_test, y_test)
print(f"Pipeline test accuracy: {score:.3f}")
print(f"Pipeline steps: {[name for name, _ in pipeline.steps]}")

# Get feature names from the encoder
encoder_step = pipeline.named_steps['encoder']
print(f"Feature names out: {encoder_step.get_feature_names_out()}")

print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 7: Inverse Transform
# ============================================================================
print("EXAMPLE 7: Inverse Transform")
print("-" * 40)

# Create encoder
encoder7 = CustomOrdinalEncoder(
    columns=['education', 'satisfaction'],
    custom_mapping={
        'education': ['High School', 'Bachelor', 'Masters', 'PhD'],
        'satisfaction': ['Low', 'Medium', 'High']
    }
)

# Fit and transform
encoder7.fit(df)
df_encoded7 = encoder7.transform(df[['education', 'satisfaction']])

print("Original data (subset):")
print(df[['education', 'satisfaction']].head(5))
print("\nEncoded data:")
print(df_encoded7[['education', 'satisfaction']].head(5))

# Inverse transform
df_decoded = encoder7.inverse_transform(df_encoded7[['education', 'satisfaction']])
print("\nDecoded data (inverse transform):")
print(df_decoded.head(5))

# Verify reconstruction
original_subset = df[['education', 'satisfaction']].head(5).reset_index(drop=True)
decoded_subset = df_decoded.head(5).reset_index(drop=True)
print(f"\nPerfect reconstruction? {original_subset.equals(decoded_subset)}")

print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 8: Mixed Data Types and Missing Values
# ============================================================================
print("EXAMPLE 8: Handling Missing Values and Mixed Types")
print("-" * 40)

# Create dataset with NaN
mixed_data = {
    'category': ['A', 'B', np.nan, 'A', 'C', 'B', None, 'A'],
    'priority': ['High', 'Medium', 'Low', np.nan, 'High', 'Medium', 'Low', 'High']
}
df_mixed = pd.DataFrame(mixed_data)

encoder8 = CustomOrdinalEncoder(
    columns=['category', 'priority'],
    custom_mapping={
        'priority': ['Low', 'Medium', 'High']  # Natural order
    },
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

df_mixed_encoded = encoder8.fit_transform(df_mixed)
print("Original data with NaN/None:")
print(df_mixed)
print("\nEncoded data (NaN preserved):")
print(df_mixed_encoded)
print("\nMapping:")
for col, mapping in encoder8.mapping_.items():
    print(f"{col}: {mapping}")

print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 9: Advanced - Partial Custom Mapping
# ============================================================================
print("EXAMPLE 9: Partial Custom Mapping")
print("-" * 40)

# Only provide mapping for some columns
partial_mapping = {
    'education': ['High School', 'Bachelor', 'Masters', 'PhD']
    # 'department' will use auto encoding
}

encoder9 = CustomOrdinalEncoder(
    columns=['education', 'department', 'size'],
    custom_mapping=partial_mapping,
    return_array=False
)

df_encoded9 = encoder9.fit_transform(df)
print("Encoded Data (Partial mapping):")
print(df_encoded9[['education', 'department', 'size']])
print("\nMapping used:")
for col, mapping in encoder9.mapping_.items():
    print(f"{col}: {mapping}")

print("\n" + "="*80 + "\n")

# ============================================================================
# EXAMPLE 10: Error Handling Demonstration
# ============================================================================
print("EXAMPLE 10: Error Handling")
print("-" * 40)

print("Test 1: Invalid column name")
try:
    encoder_err1 = CustomOrdinalEncoder(columns=['non_existent_column'])
    encoder_err1.fit(df)
except ValueError as e:
    print(f"✓ Correctly caught: {str(e)[:50]}...")

print("\nTest 2: Invalid mapping (duplicate values)")
try:
    bad_mapping = {'education': ['PhD', 'Masters', 'PhD']}  # Duplicate
    encoder_err2 = CustomOrdinalEncoder(
        columns=['education'],
        custom_mapping=bad_mapping
    )
    encoder_err2.fit(df[['education']])
except ValueError as e:
    print(f"✓ Correctly caught: {str(e)[:50]}...")

print("\nTest 3: Invalid mapping type")
try:
    bad_mapping = {'education': 'not a list or dict'}
    encoder_err3 = CustomOrdinalEncoder(
        columns=['education'],
        custom_mapping=bad_mapping
    )
    encoder_err3.fit(df[['education']])
except TypeError as e:
    print(f"✓ Correctly caught: {str(e)[:50]}...")

print("\nTest 4: Transform before fit")
try:
    encoder_err4 = CustomOrdinalEncoder(columns=['education'])
    encoder_err4.transform(df[['education']])
except RuntimeError as e:
    print(f"✓ Correctly caught: {str(e)[:50]}...")

Original Data:
     education    size satisfaction department  salary  experience
0          PhD   Small          Low         HR  150000          10
1      Masters  Medium       Medium         IT  120000           7
2     Bachelor   Large         High    Finance   90000           5
3  High School   Small          Low         IT   60000           2
4          PhD  Medium       Medium         HR  140000           9
5      Masters   Large         High    Finance  110000           6
6     Bachelor   Small          Low         IT   80000           4
7  High School  Medium       Medium         HR   55000           1
8      Masters   Large         High    Finance  100000           8
9          PhD     NaN       Medium         IT   70000           3


EXAMPLE 1: Basic Auto Encoding
----------------------------------------
Encoded Data (Auto encoding):
   education  size  satisfaction
0        3.0   2.0           1.0
1        2.0   1.0           2.0
2        0.0   0.0           0.0
3        1.0

IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

# Custom OneHot Encoder

In [18]:
import pandas as pd
import numpy as np
from typing import List, Optional, Union, Dict
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.validation import check_is_fitted

class CustomOneHotEncoder(BaseEstimator, TransformerMixin):
    """
    Custom OneHot encoder that provides flexible output options.
    
    This transformer applies one-hot encoding to specified columns with options for:
    1. Returning either pandas DataFrame or numpy array
    2. Selecting specific output columns to keep
    3. Handling unknown categories and sparse outputs
    
    Parameters
    ----------
    columns : List[str]
        The list of column names to apply one-hot encoding to.
    
    keep_columns : Optional[List[str]], default=None
        Specific one-hot encoded columns to keep in the output.
        If None, all encoded columns are returned.
    
    return_array : bool, default=False
        If True, returns numpy array. If False, returns pandas DataFrame.
    
    drop : str, default='first'
        Specifies a method to drop one of the categories per feature.
        - 'first': drop the first category (default)
        - 'if_binary': drop one category if feature is binary
        - None: retain all categories (creates k columns for k categories)
    
    sparse_output : bool, default=False
        If True, returns a sparse matrix. If False, returns dense array/DataFrame.
        Note: If return_array=False, sparse matrices are converted to dense DataFrames.
    
    handle_unknown : str, default='ignore'
        How to handle unknown categories during transform:
        - 'error': raise an error
        - 'ignore': create a row of all zeros for unknown categories
        - 'infrequent_if_exist': treat unknown as infrequent category if infrequent categories exist
    
    min_frequency : int or float, default=None
        Minimum frequency for a category to be considered frequent.
        Categories with frequency < min_frequency are grouped into infrequent categories.
    
    max_categories : int, default=None
        Maximum number of categories to keep (frequent categories only).
        Others are grouped into infrequent categories.
    
    Attributes
    ----------
    encoder_ : sklearn.preprocessing.OneHotEncoder
        The underlying sklearn OneHotEncoder instance.
    
    feature_names_in_ : np.ndarray
        Names of features seen during fit.
    
    n_features_in_ : int
        Number of features seen during fit.
    
    encoded_columns_ : List[str]
        Names of all one-hot encoded columns generated.
    
    keep_columns_idx_ : np.ndarray
        Indices of columns to keep (if keep_columns specified).
    
    categories_ : list
        Categories for each feature determined during fitting.
    
    Notes
    -----
    Author  : Rahul Shelke
    Created : 2025-07-23 
    Modified: Enhanced for production use
    
    Examples
    --------
    >>> encoder = CustomOneHotEncoder(columns=['city', 'color'])
    >>> encoder.fit(X_train)
    >>> X_encoded = encoder.transform(X_test)
    
    >>> # Keep specific encoded columns
    >>> encoder = CustomOneHotEncoder(columns=['city'], keep_columns=['city_NYC', 'city_LA'])
    
    >>> # Return numpy array
    >>> encoder = CustomOneHotEncoder(columns=['city'], return_array=True)
    """
    
    def __init__(
        self,
        columns: List[str],
        keep_columns: Optional[List[str]] = None,
        return_array: bool = False,
        drop: Optional[str] = 'first',
        sparse_output: bool = False,
        handle_unknown: str = 'ignore',
        min_frequency: Optional[Union[int, float]] = None,
        max_categories: Optional[int] = None
    ):
        self.columns = columns
        self.keep_columns = keep_columns
        self.return_array = return_array
        self.drop = drop
        self.sparse_output = sparse_output
        self.handle_unknown = handle_unknown
        self.min_frequency = min_frequency
        self.max_categories = max_categories
        
        self._validate_parameters()
    
    def _validate_parameters(self):
        """Validate initialization parameters."""
        if not isinstance(self.columns, list):
            raise TypeError(f"columns must be a list, got {type(self.columns)}")
        
        if self.keep_columns is not None:
            if not isinstance(self.keep_columns, list):
                raise TypeError(f"keep_columns must be a list or None, got {type(self.keep_columns)}")
        
        if self.drop not in [None, 'first', 'if_binary']:
            raise ValueError(f"drop must be None, 'first', or 'if_binary', got {self.drop}")
        
        if self.handle_unknown not in ['error', 'ignore', 'infrequent_if_exist']:
            raise ValueError(f"handle_unknown must be 'error', 'ignore', or 'infrequent_if_exist', got {self.handle_unknown}")
        
        if self.min_frequency is not None:
            if not isinstance(self.min_frequency, (int, float)):
                raise TypeError(f"min_frequency must be int or float, got {type(self.min_frequency)}")
            if self.min_frequency <= 0:
                raise ValueError(f"min_frequency must be > 0, got {self.min_frequency}")
        
        if self.max_categories is not None:
            if not isinstance(self.max_categories, int):
                raise TypeError(f"max_categories must be int, got {type(self.max_categories)}")
            if self.max_categories <= 0:
                raise ValueError(f"max_categories must be > 0, got {self.max_categories}")
    
    def fit(self, X: pd.DataFrame, y=None):
        """
        Fit the encoder to the data.
        
        Parameters
        ----------
        X : pd.DataFrame
            Input data to fit.
        y : None
            Ignored. Exists for compatibility.
        
        Returns
        -------
        self : CustomOneHotEncoder
            Returns the instance itself.
        """
        # Validate input
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")
        
        missing_cols = [col for col in self.columns if col not in X.columns]
        if missing_cols:
            raise ValueError(f"Columns not found in X: {missing_cols}")
        
        # Store feature names
        self.feature_names_in_ = np.array(X.columns)
        self.n_features_in_ = X.shape[1]
        
        # Initialize and fit the encoder
        self.encoder_ = OneHotEncoder(
            drop=self.drop,
            sparse_output=self.sparse_output,
            handle_unknown=self.handle_unknown,
            min_frequency=self.min_frequency,
            max_categories=self.max_categories,
            dtype=np.float64
        )
        
        # Fit encoder on selected columns
        self.encoder_.fit(X[self.columns])
        
        # Get all encoded column names
        self.encoded_columns_ = list(self.encoder_.get_feature_names_out(self.columns))
        
        # Store categories
        self.categories_ = self.encoder_.categories_
        
        # Validate and store keep_columns indices if specified
        if self.keep_columns is not None:
            missing_keep = [col for col in self.keep_columns if col not in self.encoded_columns_]
            if missing_keep:
                raise ValueError(
                    f"Columns in keep_columns not found in encoded columns: {missing_keep}. "
                    f"Available encoded columns: {self.encoded_columns_[:10]}..."
                )
            self.keep_columns_idx_ = np.array([
                self.encoded_columns_.index(col) for col in self.keep_columns
            ])
        else:
            self.keep_columns_idx_ = None
        
        return self
    
    def transform(self, X: pd.DataFrame) -> Union[pd.DataFrame, np.ndarray]:
        """
        Transform the data using the fitted encoder.
        
        Parameters
        ----------
        X : pd.DataFrame
            Input data to transform.
        
        Returns
        -------
        X_transformed : Union[pd.DataFrame, np.ndarray]
            Transformed data. Returns array if return_array=True, else DataFrame.
        """
        # Check if fit has been called
        check_is_fitted(self, 'encoder_')
        
        # Validate input
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")
        
        missing_cols = [col for col in self.columns if col not in X.columns]
        if missing_cols:
            raise ValueError(f"Columns not found in X: {missing_cols}")
        
        # Transform the data
        X_transformed = self.encoder_.transform(X[self.columns])
        
        # Apply column selection if keep_columns specified
        if self.keep_columns_idx_ is not None:
            if self.sparse_output:
                # For sparse matrices, use indexing
                X_transformed = X_transformed[:, self.keep_columns_idx_]
                selected_columns = self.keep_columns
            else:
                # For dense arrays, use regular indexing
                X_transformed = X_transformed[:, self.keep_columns_idx_]
                selected_columns = self.keep_columns
        else:
            selected_columns = self.encoded_columns_
        
        # Handle output format
        if self.return_array:
            # Convert sparse matrix to array if needed
            if hasattr(X_transformed, 'toarray'):
                return X_transformed.toarray()
            return X_transformed
        else:
            # Create DataFrame
            if hasattr(X_transformed, 'toarray'):
                # Convert sparse to dense for DataFrame
                X_transformed = X_transformed.toarray()
            
            encoded_df = pd.DataFrame(
                X_transformed,
                columns=selected_columns,
                index=X.index
            )
            
            # Preserve non-encoded columns
            other_cols = [col for col in X.columns if col not in self.columns]
            if other_cols:
                encoded_df = pd.concat([encoded_df, X[other_cols]], axis=1)
            
            return encoded_df
    
    def fit_transform(self, X: pd.DataFrame, y=None) -> Union[pd.DataFrame, np.ndarray]:
        """
        Fit and transform the data in one step.
        
        Parameters
        ----------
        X : pd.DataFrame
            Input data to fit and transform.
        y : None
            Ignored. Exists for compatibility.
        
        Returns
        -------
        X_transformed : Union[pd.DataFrame, np.ndarray]
            Transformed data.
        """
        return self.fit(X, y).transform(X)
    
    def inverse_transform(self, X: Union[pd.DataFrame, np.ndarray]) -> pd.DataFrame:
        """
        Convert one-hot encoded data back to original categories.
        
        Parameters
        ----------
        X : Union[pd.DataFrame, np.ndarray]
            One-hot encoded data to inverse transform.
        
        Returns
        -------
        X_original : pd.DataFrame
            Data with original categories.
        """
        check_is_fitted(self, 'encoder_')
        
        # Handle input based on type
        if isinstance(X, pd.DataFrame):
            # Extract only the encoded columns
            if self.keep_columns is not None:
                # We have subset of columns, need to reconstruct full encoding
                raise NotImplementedError(
                    "inverse_transform not supported when keep_columns is specified. "
                    "Use keep_columns=None for full inverse transform capability."
                )
            
            # Get encoded columns from DataFrame
            encoded_cols = [col for col in X.columns if col in self.encoded_columns_]
            X_array = X[encoded_cols].values
            result_array = self.encoder_.inverse_transform(X_array)
            
            # Create DataFrame with original column names
            result_df = pd.DataFrame(result_array, columns=self.columns, index=X.index)
            
            # Add back non-encoded columns
            other_cols = [col for col in X.columns if col not in encoded_cols]
            if other_cols:
                result_df = pd.concat([result_df, X[other_cols]], axis=1)
                
        else:
            # X is array
            if self.keep_columns is not None:
                raise NotImplementedError(
                    "inverse_transform not supported when keep_columns is specified. "
                    "Use keep_columns=None for full inverse transform capability."
                )
            
            result_array = self.encoder_.inverse_transform(X)
            result_df = pd.DataFrame(result_array, columns=self.columns)
        
        return result_df
    
    def get_feature_names_out(self, input_features=None):
        """
        Get output feature names for transformation.
        
        Parameters
        ----------
        input_features : array-like of str or None, default=None
            Input features.
        
        Returns
        -------
        feature_names_out : ndarray of str objects
            Transformed feature names.
        """
        check_is_fitted(self, 'encoder_')
        
        if self.keep_columns is not None:
            return np.array(self.keep_columns)
        else:
            return self.encoder_.get_feature_names_out(self.columns)
    
    def get_feature_names(self):
        """Get feature names (for compatibility with older sklearn versions)."""
        warnings.warn(
            "get_feature_names is deprecated. Use get_feature_names_out instead.",
            DeprecationWarning
        )
        return self.get_feature_names_out()
    
    def get_params(self, deep=True):
        """Get parameters for this estimator."""
        params = {
            'columns': self.columns,
            'keep_columns': self.keep_columns,
            'return_array': self.return_array,
            'drop': self.drop,
            'sparse_output': self.sparse_output,
            'handle_unknown': self.handle_unknown,
            'min_frequency': self.min_frequency,
            'max_categories': self.max_categories
        }
        return params
    
    def set_params(self, **params):
        """Set the parameters of this estimator."""
        for key, value in params.items():
            setattr(self, key, value)
        self._validate_parameters()
        
        # Reset fitted attributes if parameters changed
        if hasattr(self, 'encoder_'):
            del self.encoder_
        if hasattr(self, 'encoded_columns_'):
            del self.encoded_columns_
        if hasattr(self, 'keep_columns_idx_'):
            del self.keep_columns_idx_
        
        return self
    
    def get_all_encoded_columns(self):
        """
        Get all encoded column names that would be generated.
        
        Returns
        -------
        encoded_columns : List[str]
            List of all encoded column names.
        """
        check_is_fitted(self, 'encoder_')
        return self.encoded_columns_.copy()
    
    def get_encoded_column_mapping(self):
        """
        Get mapping from original categories to encoded column names.
        
        Returns
        -------
        mapping : Dict[str, List[str]]
            Dictionary mapping original column names to list of encoded column names.
        """
        check_is_fitted(self, 'encoder_')
        
        mapping = {}
        for i, col in enumerate(self.columns):
            categories = self.categories_[i]
            prefix = f"{col}_"
            encoded_cols = [
                f"{col}_{cat}" for cat in categories 
                if not (self.drop == 'first' and cat == categories[0]) and
                not (self.drop == 'if_binary' and len(categories) == 2 and cat == categories[0])
            ]
            mapping[col] = encoded_cols
        
        return mapping


class CustomOneHotEncoderWithDrop(CustomOneHotEncoder):
    """
    Extended version that can drop the original categorical columns after encoding.
    
    Parameters
    ----------
    drop_original : bool, default=True
        If True, drops the original categorical columns from the output.
    
    All other parameters are inherited from CustomOneHotEncoder.
    """
    
    def __init__(
        self,
        columns: List[str],
        keep_columns: Optional[List[str]] = None,
        return_array: bool = False,
        drop: Optional[str] = 'first',
        sparse_output: bool = False,
        handle_unknown: str = 'ignore',
        min_frequency: Optional[Union[int, float]] = None,
        max_categories: Optional[int] = None,
        drop_original: bool = True
    ):
        super().__init__(
            columns=columns,
            keep_columns=keep_columns,
            return_array=return_array,
            drop=drop,
            sparse_output=sparse_output,
            handle_unknown=handle_unknown,
            min_frequency=min_frequency,
            max_categories=max_categories
        )
        self.drop_original = drop_original
    
    def transform(self, X: pd.DataFrame) -> Union[pd.DataFrame, np.ndarray]:
        """
        Transform data and optionally drop original columns.
        """
        result = super().transform(X)
        
        if not self.return_array and self.drop_original:
            # Drop original columns from DataFrame
            if isinstance(result, pd.DataFrame):
                result = result.drop(columns=self.columns, errors='ignore')
        
        return result

### **Most Useful Ones to Implement First:**

For a practical production toolkit, I recommend implementing these in order:

**Priority 1 (Must-Have):**

1. **TargetEncoder** - Extremely useful for tree-based models

2. **FrequencyEncoder** - Simple but effective for many cases

3. **LeaveOneOutEncoder** - Prevents target leakage

**Priority 2 (Very Useful):**

4. **BinaryEncoder** - Good for high-cardinality features

5. **WeightOfEvidenceEncoder** - Essential for credit scoring

6. **CatBoostEncoder** - State-of-the-art for GBDTs

**Priority 3 (Specialized):**

7. **MEstimateEncoder** - Good alternative to target encoding

8. **CyclicEncoder** - Essential for time-related features

9. **SimilarityEncoder** - For string-based categories

### **Recommendation:**

For a production-ready categorical encoding library, implement:

1. **TargetEncoder** (for tree models)

2. **FrequencyEncoder** (simple baseline)

3. **LeaveOneOutEncoder** (prevents leakage)

4. **BinaryEncoder** (high-cardinality features)

5. **WeightOfEvidenceEncoder** (binary classification)

These cover **90%** of real-world use cases while maintaining a good balance between complexity and utility.